In [ ]:
#new code version aug 20
import csv
import re
import pandas as pd

# Paths
table1_path = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\UniSuperOptionHoldings (1).csv"
out_path    = r"D:\LinhDao\Programming\SUPERFUNdProject\UniSuper_Cleaned_final.csv"

# Final schema / headers
final_cols = [
    "Effective Date", "Fund Name", "Option Name", "Asset Class Name", "Int/Ext",
    "Name/Kind of Investment Item", "Currency", "Stock ID", "Listed Country",
    "Units Held", "% Ownership", "Address", "Value (AUD)", "Weighting"
]

# ---------------- helpers ----------------
def is_header_row(row):
    lo = [str(c or "").lower() for c in row]
    return any("name" in c for c in lo) and any("value" in c for c in lo) and any("weight" in c for c in lo)

def is_total_row(row):
    return bool(row) and str(row[0]).strip() == "TOTAL"

def parse_int_ext(text):
    s = str(text or "").strip().lower()
    if "externally managed" in s: return 1
    if "internally managed" in s: return 0
    return ""

def is_asset_class_row(row):
    if not row: return False
    first = str(row[0]).strip()
    if not first: return False
    if first.upper() in {"TABLE 1", "TOTAL", "TOTAL INVESTMENT ITEMS", "INVESTMENT OPTION NAME"}:
        return False
    if any(str(c).strip() for c in row[1:]):
        return False
    tokens = re.findall(r"[A-Za-z]+", first)
    return 1 <= len(tokens) <= 2

def map_header(row):
    header_map = {}
    for j, raw in enumerate(row):
        col = str(raw or "").strip()
        if not col:
            continue
        low = col.lower()
        if low.startswith("name"):
            header_map[j] = "Name/Kind of Investment Item"
        elif "security identifier" in low:
            header_map[j] = "Stock ID"
        elif "% of property held" in low or "% ownership" in low:
            header_map[j] = "% Ownership"
        elif "units held" in low:
            header_map[j] = "Units Held"
        elif low == "currency":
            header_map[j] = "Currency"
        elif low in {"address", "location", "listed country"}:
            header_map[j] = "Address" if low == "address" else "Listed Country"
        elif "value" in low:
            header_map[j] = "Value (AUD)"
        elif "weighting" in low:
            header_map[j] = "Weighting"
    return header_map

# ---- type rules, applied BEFORE mapping ----
def wants_string_by_raw_header(raw_header: str) -> bool:
    h = (raw_header or "").strip().lower()
    # String columns in source: name*, security identifier*, currency, % ownership (keep original text)
    return h.startswith("name") or "security identifier" in h or h == "currency" or "% ownership" in h or "% of property held" in h

def wants_numeric_by_raw_header(raw_header: str) -> bool:
    h = (raw_header or "").strip().lower()
    # Numeric columns in source: units held, any 'value' column
    return ("units held" in h) or ("value" in h)

def to_string_cell(x):
    s = "" if x is None else str(x)
    return s.strip()

_num_cleanup_re = re.compile(r"[,\s]")
def to_number_cell(x):
    if x is None:
        return None
    s = str(x).strip()
    if s == "":
        return None
    # Handle (1,234.56) negatives
    neg = s.startswith("(") and s.endswith(")")
    s = s.strip("()")
    # Remove currency symbols and commas/spaces
    s = s.replace("$", "").replace("AUD", "").replace("A$", "")
    s = _num_cleanup_re.sub("", s)
    try:
        val = float(s)
        return -val if neg else val
    except:
        return None  # if it doesn't parse, treat as missing numeric

def clean_weight(val):
    if val is None:
        return None
    s = str(val).strip().replace("%", "")
    s = _num_cleanup_re.sub("", s)
    if s == "":
        return None
    try:
        return float(s) / 100.0
    except:
        return None

# --- Load TABLE 1 csv ---
with open(table1_path, "r", encoding="cp1252", newline="") as f:
    rows = list(csv.reader(f))

all_out = []
i, n = 0, len(rows)

while i < n:
    row = rows[i]
    if is_asset_class_row(row):
        # Title-case the asset class name
        asset_class_name = str(row[0]).strip().title()
        int_ext = ""

        # Find header row, capturing any Int/Ext line before it
        k = i + 1
        header_idx = None
        while k < n:
            text_line = " ".join(str(c).strip() for c in rows[k] if str(c).strip())
            if int_ext == "":
                ie = parse_int_ext(text_line)
                if ie != "":
                    int_ext = ie
            if is_header_row(rows[k]):
                header_idx = k
                break
            if is_total_row(rows[k]):
                break
            k += 1

        if header_idx is None:
            i += 1
            continue

        header_row = rows[header_idx]
        header_map = map_header(header_row)

        # Build a type-intent map keyed by raw column index, based on RAW HEADERS
        type_intent = {}
        for raw_idx, tgt in header_map.items():
            raw_header = header_row[raw_idx] if raw_idx < len(header_row) else ""
            if wants_string_by_raw_header(raw_header):
                type_intent[raw_idx] = "str"
            elif wants_numeric_by_raw_header(raw_header):
                type_intent[raw_idx] = "num"
            else:
                # default safer: treat as string
                type_intent[raw_idx] = "str"

        # Collect rows up to and including TOTAL
        j = header_idx + 1
        while j < n:
            r = rows[j]
            out = {c: "" for c in final_cols}
            out.update({
                "Effective Date": pd.Timestamp(2024, 12, 31),  # datetime, not string
                "Fund Name": "UniSuper",
                "Option Name": "Balanced",
                "Asset Class Name": asset_class_name,
                "Int/Ext": int_ext,
            })

            # Coerce types BEFORE mapping
            for raw_idx, tgt in header_map.items():
                if raw_idx >= len(r):
                    continue
                raw_val = r[raw_idx]

                # DO NOT TOUCH "% Ownership" (keep original text)
                if tgt == "% Ownership":
                    out[tgt] = to_string_cell(raw_val)
                    continue

                if type_intent.get(raw_idx) == "num":
                    coerced = to_number_cell(raw_val)
                    out[tgt] = coerced  # keep numeric (float) or None
                else:
                    out[tgt] = to_string_cell(raw_val)

            all_out.append(out)

            if is_total_row(r):
                j += 1
                break
            j += 1

        i = j
        continue
    i += 1

# Build DataFrame
df_out = pd.DataFrame(all_out, columns=final_cols)

# --- Post-processing ---

# 1) Int/Ext: fill blanks with 1 (int) — Option A fix (nullable Int64 -> fill -> int)
df_out["Int/Ext"] = (
    pd.Series(df_out["Int/Ext"])
    .replace("", pd.NA)
    .astype("Int64")   # explicit nullable integer dtype
    .fillna(1)
    .astype(int)       # strict int after fill
)

# 2) Stock ID -> Listed Country (first 2 chars), and trim Stock ID
def split_stockid(val):
    if pd.isna(val) or str(val).strip() == "":
        return ("", "")
    s = str(val).strip()
    if len(s) > 2:
        return (s, s[:2])
    return (s, "")

# Ensure Stock ID is string before splitting
df_out["Stock ID"] = df_out["Stock ID"].astype("string").fillna("")
df_out["Stock ID"], df_out["Listed Country"] = zip(*df_out["Stock ID"].map(split_stockid))

# 3) Weighting: remove % and divide by 100 (keep as float)
df_out["Weighting"] = df_out["Weighting"].apply(clean_weight)

# 4) Convert literal "TOTAL" names to "Sub Total"
name_col = df_out["Name/Kind of Investment Item"].astype("string").fillna("")
df_out.loc[name_col.str.strip().eq("TOTAL"), "Name/Kind of Investment Item"] = "Sub Total"

# 5) Enforce final dtypes
# - Effective Date: keep as datetime64[ns]
# - % Ownership: keep original text exactly (string)
string_cols = [
    "Fund Name", "Option Name", "Asset Class Name",
    "Name/Kind of Investment Item", "Currency", "Stock ID", "Listed Country", "Address",
    "% Ownership",
]
numeric_cols = ["Units Held", "Value (AUD)", "Weighting"]

for c in string_cols:
    df_out[c] = df_out[c].astype("string").fillna("")

for c in numeric_cols:
    df_out[c] = pd.to_numeric(df_out[c], errors="coerce")

# Ensure Effective Date is datetime
df_out["Effective Date"] = pd.to_datetime(df_out["Effective Date"])

# Save — export Effective Date as "Dec 31 2024"
df_out.to_csv(out_path, index=False, encoding="cp1252", date_format="%b %d %Y")
print(f"✅ Cleaned file saved to: {out_path} — rows: {len(df_out)}")


✅ Cleaned file saved to: D:\LinhDao\Programming\SUPERFUNdProject\UniSuper_Cleaned_final.csv — rows: 3541
